# BLEVE Blast Overpressure Prediction

This notebook predicts peak pressure generated by Boiling Liquid Expanding Vapour Explosions (BLEVEs) using the supplied training and competition test datasets.

> **Data placement:** Locally, put the original course files at `data/train.csv` and `data/test.csv`. In Google Colab, the dataset setup cell prompts you to upload missing required files. An optional `sample_prediction.csv` can be uploaded to enforce the competition submission format.

> **Evaluation note:** `test.csv` has no target values, so MAPE/R² cannot be calculated on it. The labelled internal holdout is used for local test metrics; the Kaggle score is measured after uploading `outputs/prediction.csv`.

## 1. Environment setup and dependencies

This notebook supports both **Google Colab** and a **local Python/Jupyter environment**.

- In Colab, run the dependency-check cell. The dataset loader will ask you to upload `train.csv` and `test.csv` if they are not already available.
- Locally, install `requirements.txt`, place the authorised CSV files in `data/`, and run the notebook from the project folder.
- `sample_prediction.csv` is optional; it is used only to match the expected competition submission columns and row order.


In [ ]:
# Install only packages that are not already available in this runtime.
# For a local environment, you can instead run: python -m pip install -r requirements.txt
import importlib.util
import subprocess
import sys

PACKAGE_SPECS = {
    "numpy": "numpy>=1.24",
    "pandas": "pandas>=2.0",
    "matplotlib": "matplotlib>=3.7",
    "sklearn": "scikit-learn>=1.3",
    "xgboost": "xgboost>=2.0",
}
missing_packages = [spec for module, spec in PACKAGE_SPECS.items()
                    if importlib.util.find_spec(module) is None]
if missing_packages:
    print("Installing missing packages:", missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("All required machine-learning packages are already installed.")


In [ ]:
from pathlib import Path
import json
import warnings
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
    make_scorer,
)
from sklearn.model_selection import (
    GroupKFold,
    GroupShuffleSplit,
    KFold,
    RandomizedSearchCV,
    train_test_split,
)
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVR
from xgboost import XGBRegressor

warnings.filterwarnings("default")

RANDOM_STATE = 42
TARGET = "Target Pressure (bar)"
ID_COLUMN = "ID"

# Detect Google Colab. Locally, use the repository folder containing this notebook.
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    colab_files = None
    IN_COLAB = False


def find_project_root(start_directory):
    """Find the project folder when Jupyter is launched from a child directory."""
    start_directory = Path(start_directory).resolve()
    for candidate in (start_directory, *start_directory.parents):
        if (candidate / "requirements.txt").exists() and (candidate / "main.ipynb").exists():
            return candidate
    return start_directory


# Colab normally runs in /content. Local Jupyter should be launched from the project root.
PROJECT_ROOT = Path.cwd().resolve() if IN_COLAB else find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_prediction.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Cleaning / validation configuration.
ROW_MISSING_THRESHOLD = 0.40  # remove labelled records with >= 40% missing predictors
REMOVE_TARGET_IQR_OUTLIERS = False  # Retain potentially meaningful extreme BLEVE pressures by default
IQR_MULTIPLIER = 1.5

# Search configuration. Increase N_ITER_SEARCH for a broader search if compute permits.
CV_FOLDS = 2
N_ITER_SEARCH = 2
SEARCH_N_JOBS = 1

np.random.seed(RANDOM_STATE)
print("Configuration loaded.")

## 2. Locate and load the course datasets

The next cell checks `data/` and the project root. In Google Colab, it opens an upload dialog if `train.csv` or `test.csv` is missing. Locally, place the files in `data/` (the project root is also checked for convenience).


In [ ]:
def normalise_upload_filename(filename):
    """Normalise browser-upload filenames to predictable CSV names."""
    name = Path(str(filename).replace("\\", "/")).name.strip().lower()
    return "_".join(name.replace("-", " ").split())


def find_existing_data_file(canonical_name):
    """Find a required data file in data/ or the project root, case-insensitively."""
    search_directories = [DATA_DIR, PROJECT_ROOT]
    for directory in search_directories:
        if not directory.exists():
            continue
        for candidate in directory.iterdir():
            if candidate.is_file() and normalise_upload_filename(candidate.name) == canonical_name:
                return candidate
    return None


# Move files uploaded to /content (or placed beside the notebook) into data/.
for expected_name in ("train.csv", "test.csv", "sample_prediction.csv"):
    existing_file = find_existing_data_file(expected_name)
    destination = DATA_DIR / expected_name
    if existing_file is not None and existing_file.resolve() != destination.resolve():
        shutil.copy2(existing_file, destination)

missing_required = [name for name in ("train.csv", "test.csv")
                    if not (DATA_DIR / name).is_file()]

if missing_required and IN_COLAB:
    print("Upload the required course datasets: " + ", ".join(missing_required))
    print("You may also select sample_prediction.csv if it was supplied.")
    uploaded_files = colab_files.upload()
    for uploaded_name, file_bytes in uploaded_files.items():
        canonical_name = normalise_upload_filename(uploaded_name)
        if canonical_name in {"train.csv", "test.csv", "sample_prediction.csv"}:
            (DATA_DIR / canonical_name).write_bytes(file_bytes)

missing_required = [name for name in ("train.csv", "test.csv")
                    if not (DATA_DIR / name).is_file()]
if missing_required:
    raise FileNotFoundError(
        "Missing required dataset(s): " + ", ".join(missing_required) + "\n"
        + "Place them in the project data/ folder. In Google Colab, rerun this cell and upload both files."
    )

if not SAMPLE_SUBMISSION_PATH.exists():
    print("Optional sample_prediction.csv not found; a standard ID/target submission will be created.")
else:
    print("Optional sample submission template found.")

print("Execution environment:", "Google Colab" if IN_COLAB else "Local Python/Jupyter")
print("Dataset directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)


In [ ]:
for required_path in (TRAIN_PATH, TEST_PATH):
    if not required_path.exists():
        raise FileNotFoundError(
            f"Missing {required_path}. Download the original COMP3010 train.csv and "
            "test.csv files and place them in the data/ folder."
        )

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

print(f"Training data: {train_raw.shape}")
print(f"Competition test data: {test_raw.shape}")
display(train_raw.head())
display(test_raw.head())

if TARGET not in train_raw.columns:
    raise ValueError(f"Training data must contain target column: {TARGET}")
if ID_COLUMN not in train_raw.columns or ID_COLUMN not in test_raw.columns:
    raise ValueError(f"Both datasets must contain the identifier column: {ID_COLUMN}")

missing_test_columns = sorted(set(train_raw.columns) - {TARGET} - set(test_raw.columns))
if missing_test_columns:
    raise ValueError(f"Columns missing from test.csv: {missing_test_columns}")

## 3. Inspect data quality and standardise categories

In [ ]:
def normalise_status(value):
    """Map observed spelling/format variants to the two specified Status labels."""
    if pd.isna(value):
        return np.nan

    text = " ".join(str(value).strip().lower().replace("_", " ").replace("-", " ").split())
    compact = text.replace(" ", "")
    aliases = {
        "subcooled": "Subcooled",
        "subcooledstate": "Subcooled",
        "subcool": "Subcooled",
        "subcoled": "Subcooled",
        "superheated": "Superheated",
        "superheatedstate": "Superheated",
        "superheat": "Superheated",
        "saperheated": "Superheated",
    }
    return aliases.get(compact, aliases.get(text, text.title()))


def normalise_category_column(series, column_name):
    """Trim categorical text and format numeric category identifiers consistently."""
    def convert(value):
        if pd.isna(value):
            return np.nan
        if isinstance(value, (int, np.integer)):
            return str(int(value))
        if isinstance(value, (float, np.floating)) and np.isfinite(value):
            return str(int(value)) if float(value).is_integer() else f"{value:g}"
        return " ".join(str(value).strip().split())

    if column_name == "Status":
        return series.map(normalise_status)
    return series.map(convert)


def standardise_categories(frame):
    frame = frame.copy()
    categorical_candidates = [
        col for col in ["Status", "Sensor ID", "Sensor Position Side"]
        if col in frame.columns
    ]
    for col in categorical_candidates:
        frame[col] = normalise_category_column(frame[col], col)
    return frame


train = standardise_categories(train_raw)
test = standardise_categories(test_raw)

if "Status" in train.columns:
    print("Status values after normalisation (training):")
    display(train["Status"].value_counts(dropna=False).rename_axis("Status").to_frame("Count"))
    known_statuses = {"Subcooled", "Superheated"}
    unknown_statuses = sorted(set(train["Status"].dropna().unique()) - known_statuses)
    if unknown_statuses:
        raise ValueError(f"Unrecognised Status values need manual review: {unknown_statuses}")

print("\nMissing values by training column:")
missing_summary = train.isna().sum().sort_values(ascending=False)
display(missing_summary[missing_summary > 0].to_frame("Missing count"))
print("Duplicate rows before removal:", train.drop(columns=[ID_COLUMN], errors="ignore").duplicated().sum())


## 4. Clean labelled records and remove high-missingness rows

In [ ]:
# Keep the identifier separately for the competition submission.
test_ids = test[ID_COLUMN].copy()

# Convert the target to numeric and remove records with missing or invalid target values.
train[TARGET] = pd.to_numeric(train[TARGET], errors="coerce")
missing_target_count = int(train[TARGET].isna().sum())
train = train.dropna(subset=[TARGET]).copy()

# MAPE and log-target regression require positive target values for this task.
non_positive_target_count = int((train[TARGET] <= 0).sum())
if non_positive_target_count:
    raise ValueError(
        f"Found {non_positive_target_count} non-positive target values. Inspect these rows "
        "before modelling; do not silently discard them."
    )

# Remove duplicate observations while ignoring ID, which is only a row identifier.
duplicate_subset = [c for c in train.columns if c != ID_COLUMN]
duplicate_count = int(train.duplicated(subset=duplicate_subset).sum())
train = train.drop_duplicates(subset=duplicate_subset).copy()

# Remove labelled rows with a high proportion of missing predictor values.
predictor_columns_raw = [c for c in train.columns if c not in {ID_COLUMN, TARGET}]
row_missing_fraction = train[predictor_columns_raw].isna().mean(axis=1)
high_missing_mask = row_missing_fraction >= ROW_MISSING_THRESHOLD
high_missing_count = int(high_missing_mask.sum())
train = train.loc[~high_missing_mask].copy()

# Keep all competition test records. Remaining missing predictors are imputed inside pipelines.
print(f"Rows removed for missing target: {missing_target_count}")
print(f"Duplicate rows removed (ID ignored): {duplicate_count}")
print(f"Rows removed with >= {ROW_MISSING_THRESHOLD:.0%} missing predictors: {high_missing_count}")
print(f"Clean labelled rows retained: {len(train):,}")
print(f"Competition test rows retained: {len(test):,}")

if len(train) < 100:
    raise ValueError("Too few labelled rows remain after cleaning to build a reliable model.")

## 5. Feature engineering

In [ ]:
def safe_divide(numerator, denominator):
    """Divide while converting zero denominators and infinities to NaN."""
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")
    result = numerator / denominator.replace(0, np.nan)
    return result.replace([np.inf, -np.inf], np.nan)


def add_features(frame, liquid_ratio_scale):
    """Create deterministic physical features using columns available in the dataset."""
    frame = frame.copy()

    def has(*columns):
        return all(column in frame.columns for column in columns)

    if has("Tank Width (m)", "Tank Length (m)"):
        frame["Tank_Ratio"] = safe_divide(frame["Tank Width (m)"], frame["Tank Length (m)"])

    if has("Tank Width (m)", "Tank Length (m)", "Tank Height (m)"):
        frame["Tank_Volume"] = (
            pd.to_numeric(frame["Tank Width (m)"], errors="coerce")
            * pd.to_numeric(frame["Tank Length (m)"], errors="coerce")
            * pd.to_numeric(frame["Tank Height (m)"], errors="coerce")
        )

    if has("Vapour Temperature (K)", "Liquid Temperature (K)"):
        vapour = pd.to_numeric(frame["Vapour Temperature (K)"], errors="coerce")
        liquid = pd.to_numeric(frame["Liquid Temperature (K)"], errors="coerce")
        frame["Temp_Difference_K"] = vapour - liquid
        frame["Temp_Ratio"] = safe_divide(vapour, liquid)

    if has("Tank Failure Pressure (bar)", "Liquid Critical Pressure (bar)"):
        frame["Pressure_Ratio"] = safe_divide(
            frame["Tank Failure Pressure (bar)"], frame["Liquid Critical Pressure (bar)"]
        )

    if has("Obstacle Width (m)", "Obstacle Height (m)", "Obstacle Thickness (m)"):
        frame["Obstacle_Volume"] = (
            pd.to_numeric(frame["Obstacle Width (m)"], errors="coerce")
            * pd.to_numeric(frame["Obstacle Height (m)"], errors="coerce")
            * pd.to_numeric(frame["Obstacle Thickness (m)"], errors="coerce")
        )

    if has("Sensor Position x", "Sensor Position y", "Sensor Position z"):
        x = pd.to_numeric(frame["Sensor Position x"], errors="coerce")
        y = pd.to_numeric(frame["Sensor Position y"], errors="coerce")
        z = pd.to_numeric(frame["Sensor Position z"], errors="coerce")
        frame["Sensor_Distance"] = np.sqrt(x.pow(2) + y.pow(2) + z.pow(2))
        frame["Distance_xy"] = np.sqrt(x.pow(2) + y.pow(2))
        if "BLEVE Height (m)" in frame.columns:
            frame["Sensor_z_relative"] = z - pd.to_numeric(frame["BLEVE Height (m)"], errors="coerce")

    if has("Liquid Ratio (%)", "Tank_Volume"):
        # The supplied sample shows values such as 0.42. Treat ratios in [0, 1]
        # as fractions; if the dataset is expressed on a 0-100 scale, divide by 100.
        liquid_fraction = pd.to_numeric(frame["Liquid Ratio (%)"], errors="coerce") / liquid_ratio_scale
        frame["Liquid_Volume"] = liquid_fraction * frame["Tank_Volume"]

    if has("Tank Failure Pressure (bar)", "Tank_Volume"):
        frame["Energy_Proxy"] = (
            pd.to_numeric(frame["Tank Failure Pressure (bar)"], errors="coerce")
            * frame["Tank_Volume"]
        )

    if has("Tank Failure Pressure (bar)", "Sensor_Distance"):
        frame["Pressure_Distance"] = safe_divide(
            frame["Tank Failure Pressure (bar)"], frame["Sensor_Distance"] + 1.0
        )

    if has("Energy_Proxy", "Sensor_Distance"):
        frame["Energy_Distance"] = safe_divide(
            frame["Energy_Proxy"], frame["Sensor_Distance"] + 1.0
        )
        frame["Volume_Distance"] = safe_divide(
            frame["Tank_Volume"], frame["Sensor_Distance"] + 1.0
        )

    for source, destination in [
        ("Sensor_Distance", "Log_Distance"),
        ("Tank_Volume", "Log_Tank_Volume"),
        ("Energy_Proxy", "Log_Energy"),
    ]:
        if source in frame.columns:
            frame[destination] = np.log1p(pd.to_numeric(frame[source], errors="coerce").clip(lower=0))

    if has("BLEVE Height (m)", "Tank Height (m)"):
        frame["Height_Ratio"] = safe_divide(
            frame["BLEVE Height (m)"], frame["Tank Height (m)"]
        )

    if has("Tank Failure Pressure (bar)", "Vapour Temperature (K)"):
        frame["Pressure_Temp"] = (
            pd.to_numeric(frame["Tank Failure Pressure (bar)"], errors="coerce")
            * pd.to_numeric(frame["Vapour Temperature (K)"], errors="coerce")
        )

    return frame.replace([np.inf, -np.inf], np.nan)

# Detect whether Liquid Ratio (%) is supplied as a fraction (0-1) or percentage (0-100).
liquid_ratio_values = pd.to_numeric(train["Liquid Ratio (%)"], errors="coerce").dropna() if "Liquid Ratio (%)" in train.columns else pd.Series(dtype=float)
liquid_ratio_scale = 100.0 if not liquid_ratio_values.empty and liquid_ratio_values.quantile(0.95) > 1.0 else 1.0
print("Liquid ratio conversion scale:", liquid_ratio_scale)

train_features = add_features(train.drop(columns=[TARGET]), liquid_ratio_scale)
test_features = add_features(test.drop(columns=[ID_COLUMN], errors="ignore"), liquid_ratio_scale)

# The identifier is not a predictive feature.
X = train_features.drop(columns=[ID_COLUMN], errors="ignore")
y = train[TARGET].astype(float).reset_index(drop=True)
X = X.reset_index(drop=True)
X_test = test_features.reset_index(drop=True)

# Keep exactly the training feature columns in the same order for test inference.
missing_from_test = sorted(set(X.columns) - set(X_test.columns))
if missing_from_test:
    raise ValueError(f"Features missing from test data after feature engineering: {missing_from_test}")
X_test = X_test[X.columns].copy()

# Drop columns that are entirely missing in the labelled training data.
# This prevents imputers from receiving a feature with no observed training values.
all_missing_columns = [column for column in X.columns if X[column].isna().all()]
if all_missing_columns:
    print("Dropping entirely missing training features:", all_missing_columns)
    X = X.drop(columns=all_missing_columns)
    X_test = X_test.drop(columns=all_missing_columns)

print("Feature matrix:", X.shape)
print("Competition features:", X_test.shape)

## 6. Create training, validation and internal test partitions

In [ ]:
def create_scenario_groups(ids, sensor_ids, expected_sensors=27):
    """Derive physical scenario groups from the verified 27-row sensor ID sequence.

    The supplied specification describes 27 sensors per obstacle. In these files,
    Sensor ID equals (ID % 27) + 1 for every row with both values present.
    Consequently, floor(ID / 27) identifies the originating scenario. Rows with a
    missing ID receive unique groups so they cannot be joined to an unrelated case.
    """
    id_values = pd.to_numeric(ids, errors="coerce").reset_index(drop=True)
    sensor_values = pd.to_numeric(sensor_ids, errors="coerce").reset_index(drop=True)
    valid = id_values.notna() & sensor_values.notna()
    if not valid.any():
        print("Scenario grouping unavailable: no valid ID/sensor pairs. Using seeded random splits.")
        return None

    expected_sensor = (id_values.loc[valid].astype(np.int64) % expected_sensors) + 1
    alignment = (sensor_values.loc[valid].astype(np.int64) == expected_sensor).mean()
    if alignment < 0.99:
        print(f"ID/sensor sequence check failed ({alignment:.1%} aligned); using seeded random splits.")
        return None

    groups = []
    for row_number, identifier in enumerate(id_values):
        if pd.isna(identifier):
            groups.append(f"missing-id-row-{row_number}")
        else:
            groups.append(f"scenario-{int(identifier) // expected_sensors}")

    result = pd.Series(groups, name="ScenarioGroup", dtype="string")
    print(f"Verified ID/sensor alignment: {alignment:.1%}.")
    print(f"Scenario-aware splitting enabled: {result.nunique():,} scenarios detected.")
    return result


def split_three_ways(X_data, y_data, groups=None):
    """Create approximately 70/15/15 train/validation/internal-test partitions."""
    indices = np.arange(len(X_data))

    if groups is not None:
        first_split = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
        train_idx, temporary_idx = next(first_split.split(X_data, y_data, groups=groups))

        X_temp = X_data.iloc[temporary_idx]
        y_temp = y_data.iloc[temporary_idx]
        groups_temp = groups.iloc[temporary_idx]
        second_split = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_STATE + 1)
        val_relative, test_relative = next(
            second_split.split(X_temp, y_temp, groups=groups_temp)
        )
        val_idx = temporary_idx[val_relative]
        internal_test_idx = temporary_idx[test_relative]
    else:
        train_idx, temporary_idx = train_test_split(
            indices, test_size=0.30, random_state=RANDOM_STATE
        )
        val_idx, internal_test_idx = train_test_split(
            temporary_idx, test_size=0.50, random_state=RANDOM_STATE + 1
        )

    result = (
        X_data.iloc[train_idx].reset_index(drop=True),
        X_data.iloc[val_idx].reset_index(drop=True),
        X_data.iloc[internal_test_idx].reset_index(drop=True),
        y_data.iloc[train_idx].reset_index(drop=True),
        y_data.iloc[val_idx].reset_index(drop=True),
        y_data.iloc[internal_test_idx].reset_index(drop=True),
        groups.iloc[train_idx].reset_index(drop=True) if groups is not None else None,
    )
    return result

scenario_groups = create_scenario_groups(
    train[ID_COLUMN].reset_index(drop=True),
    train["Sensor ID"].reset_index(drop=True),
)
(
    X_train_raw, X_val, X_internal_test,
    y_train_raw, y_val, y_internal_test,
    groups_train_raw,
) = split_three_ways(X, y, scenario_groups)

print(f"Training rows before target-outlier policy: {len(X_train_raw):,}")
print(f"Validation rows: {len(X_val):,}")
print(f"Internal test rows: {len(X_internal_test):,}")

# Report target IQR outliers, but retain them by default: high pressures can be
# physically meaningful. Set REMOVE_TARGET_IQR_OUTLIERS=True only for comparison.
q1 = y_train_raw.quantile(0.25)
q3 = y_train_raw.quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - IQR_MULTIPLIER * iqr
upper_bound = q3 + IQR_MULTIPLIER * iqr
train_target_outlier_mask = (y_train_raw < lower_bound) | (y_train_raw > upper_bound)
print(f"Training target IQR bounds: [{lower_bound:.6f}, {upper_bound:.6f}]")
print(f"Potential target outliers in training partition: {int(train_target_outlier_mask.sum()):,}")

if REMOVE_TARGET_IQR_OUTLIERS:
    keep_mask = ~train_target_outlier_mask
    X_train = X_train_raw.loc[keep_mask].reset_index(drop=True)
    y_train = y_train_raw.loc[keep_mask].reset_index(drop=True)
    groups_train = groups_train_raw.loc[keep_mask].reset_index(drop=True) if groups_train_raw is not None else None
else:
    X_train = X_train_raw.copy()
    y_train = y_train_raw.copy()
    groups_train = groups_train_raw.copy() if groups_train_raw is not None else None

print(f"Training rows used: {len(X_train):,}")
print("Validation and internal-test targets were not removed or altered.")

if len(X_train) < 100:
    raise ValueError("Too few training rows remain after outlier handling. Review the policy.")
if (y_train <= 0).any() or (y_val <= 0).any() or (y_internal_test <= 0).any():
    raise ValueError("Target pressure must be positive for log-target regression and MAPE evaluation.")


### Outlier treatment

The supplied training data contains high target-pressure observations. These may represent physically meaningful severe BLEVE scenarios, so the default workflow **does not delete target outliers**. The target is log-transformed during fitting to reduce skew. A configurable target-IQR exclusion option remains available for a sensitivity experiment, but it should not be enabled automatically.

For numeric predictors, `IQRClipper` is fitted inside each preprocessing pipeline. Its bounds are learned only from the corresponding training fold, then applied to that fold's validation data. This addresses extreme predictor values without leaking information from validation or internal-test partitions.


## 7. Build leakage-safe preprocessing pipelines

In [ ]:
FORCE_CATEGORICAL = [
    col for col in ["Status", "Sensor ID", "Sensor Position Side"]
    if col in X_train.columns
]

# Convert forced categorical identifiers to strings in all partitions.
def cast_categorical(frame, columns):
    frame = frame.copy()
    for col in columns:
        frame[col] = frame[col].astype("string")
    return frame

X_train = cast_categorical(X_train, FORCE_CATEGORICAL)
X_val = cast_categorical(X_val, FORCE_CATEGORICAL)
X_internal_test = cast_categorical(X_internal_test, FORCE_CATEGORICAL)
X_test = cast_categorical(X_test, FORCE_CATEGORICAL)

numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

print("Numeric features:", len(numeric_features))
print("Categorical features:", categorical_features)

try:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:  # compatibility with older scikit-learn releases
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse=True)


class IQRClipper(BaseEstimator, TransformerMixin):
    """Clip numeric feature outliers using IQR bounds learned on the fit partition."""

    def __init__(self, factor=1.5):
        self.factor = factor

    def fit(self, X, y=None):
        values = np.asarray(X, dtype=float)
        q1 = np.nanpercentile(values, 25, axis=0)
        q3 = np.nanpercentile(values, 75, axis=0)
        spread = q3 - q1
        self.lower_bounds_ = q1 - self.factor * spread
        self.upper_bounds_ = q3 + self.factor * spread
        return self

    def transform(self, X):
        values = np.asarray(X, dtype=float)
        return np.clip(values, self.lower_bounds_, self.upper_bounds_)


def make_preprocessor(scale_numeric):
    numeric_steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("iqr_clip", IQRClipper(factor=IQR_MULTIPLIER)),
    ]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_pipeline = Pipeline(numeric_steps)
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", clone(one_hot)),
    ])

    transformers = []
    if numeric_features:
        transformers.append(("numeric", numeric_pipeline, numeric_features))
    if categorical_features:
        transformers.append(("categorical", categorical_pipeline, categorical_features))

    return ColumnTransformer(transformers=transformers, remainder="drop")


def make_model_pipeline(estimator, scale_numeric):
    target_model = TransformedTargetRegressor(
        regressor=estimator,
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    )
    return Pipeline([
        ("preprocessor", make_preprocessor(scale_numeric)),
        ("target_model", target_model),
    ])

## 8. Define model families and hyperparameter search spaces

In [ ]:
# Linear Regression is retained as an untuned baseline. Ridge is the tuned linear model.
model_specs = {
    "Linear Regression (baseline)": {
        "estimator": LinearRegression(),
        "scale": True,
        "params": None,
    },
    "Ridge Regression": {
        "estimator": Ridge(),
        "scale": True,
        "params": {
            "target_model__regressor__alpha": [0.01, 0.1, 1.0, 10.0, 100.0],
        },
    },
    "Linear SVR": {
        "estimator": LinearSVR(random_state=RANDOM_STATE, max_iter=50000, dual=True, loss="squared_epsilon_insensitive", tol=1e-3),
        "scale": True,
        "params": {
            "target_model__regressor__C": [0.01, 0.1, 1.0, 10.0],
            "target_model__regressor__epsilon": [0.01, 0.05, 0.1],
        },
    },
    "MLP Neural Network": {
        "estimator": MLPRegressor(
            hidden_layer_sizes=(32,),
            max_iter=180,
            early_stopping=True,
            random_state=RANDOM_STATE,
            n_iter_no_change=15,
        ),
        "scale": True,
        "params": {
            "target_model__regressor__hidden_layer_sizes": [(32,), (64,)],
            "target_model__regressor__alpha": [0.001, 0.01],
        },
    },
    "Random Forest": {
        "estimator": RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1),
        "scale": False,
        "params": {
            "target_model__regressor__n_estimators": [20, 35],
            "target_model__regressor__max_depth": [None, 10, 16],
            "target_model__regressor__min_samples_leaf": [1, 2, 4],
            "target_model__regressor__max_features": [0.7, 1.0, "sqrt"],
        },
    },
    "XGBoost": {
        "estimator": XGBRegressor(
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=1,
            tree_method="hist",
        ),
        "scale": False,
        "params": {
            "target_model__regressor__n_estimators": [60, 100],
            "target_model__regressor__learning_rate": [0.02, 0.04, 0.06],
            "target_model__regressor__max_depth": [3, 4, 5],
            "target_model__regressor__min_child_weight": [1, 2, 3, 5],
            "target_model__regressor__subsample": [0.7, 0.85, 1.0],
            "target_model__regressor__colsample_bytree": [0.7, 0.85, 1.0],
            "target_model__regressor__reg_alpha": [0.0, 0.1, 0.5],
            "target_model__regressor__reg_lambda": [1.0, 2.0, 4.0],
        },
    },
}

print("Models to evaluate:", list(model_specs))
print("The search covers linear, support-vector, neural-network and tree-based model families.")

## 9. Define evaluation metrics and cross-validation strategy

In [ ]:
def safe_mape(y_true, y_pred):
    """MAPE with a small denominator floor; target values are separately checked as positive."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denominator = np.maximum(np.abs(y_true), 1e-8)
    return float(np.mean(np.abs((y_true - y_pred) / denominator)))


def calculate_metrics(y_true, y_pred):
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 0.0)
    y_true = np.asarray(y_true, dtype=float)
    return {
        "MAPE": safe_mape(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }

mape_scorer = make_scorer(safe_mape, greater_is_better=False)

if groups_train is not None and groups_train.nunique() >= CV_FOLDS:
    cv_strategy = GroupKFold(n_splits=CV_FOLDS)
    cv_groups = groups_train
    print(f"Cross-validation: GroupKFold({CV_FOLDS}) to keep scenario groups together.")
else:
    cv_strategy = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    cv_groups = None
    print(f"Cross-validation: shuffled KFold({CV_FOLDS}).")

## 10. Train and tune the candidate models

In [ ]:
fitted_models = {}
comparison_rows = []
search_details = {}

for model_name, specification in model_specs.items():
    print(f"\n{'=' * 78}\nTraining: {model_name}\n{'=' * 78}")
    pipeline = make_model_pipeline(
        estimator=clone(specification["estimator"]),
        scale_numeric=specification["scale"],
    )

    if specification["params"] is None:
        fitted_model = pipeline.fit(X_train, y_train)
        best_params = {"method": "baseline; no tunable hyperparameters"}
        cv_mape = np.nan
    else:
        search = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=specification["params"],
            n_iter=N_ITER_SEARCH,
            scoring=mape_scorer,
            cv=cv_strategy,
            random_state=RANDOM_STATE,
            n_jobs=SEARCH_N_JOBS,
            refit=True,
            verbose=1,
            error_score="raise",
        )
        if cv_groups is not None:
            search.fit(X_train, y_train, groups=cv_groups)
        else:
            search.fit(X_train, y_train)
        fitted_model = search.best_estimator_
        best_params = search.best_params_
        cv_mape = -float(search.best_score_)
        search_details[model_name] = {
            "best_params": best_params,
            "best_cv_mape": cv_mape,
        }

    fitted_models[model_name] = fitted_model

    train_prediction = np.maximum(fitted_model.predict(X_train), 0.0)
    validation_prediction = np.maximum(fitted_model.predict(X_val), 0.0)
    train_metrics = calculate_metrics(y_train, train_prediction)
    validation_metrics = calculate_metrics(y_val, validation_prediction)

    comparison_rows.append({
        "Model": model_name,
        "CV MAPE": cv_mape,
        "Train MAPE": train_metrics["MAPE"],
        "Validation MAPE": validation_metrics["MAPE"],
        "Validation R2": validation_metrics["R2"],
        "Validation MAE": validation_metrics["MAE"],
        "Validation RMSE": validation_metrics["RMSE"],
        "Best parameters": json.dumps(best_params, default=str),
    })

model_results = pd.DataFrame(comparison_rows).sort_values(
    "Validation MAPE", ascending=True
).reset_index(drop=True)

display(model_results)
model_results.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
with open(OUTPUT_DIR / "hyperparameter_search.json", "w", encoding="utf-8") as file:
    json.dump(search_details, file, indent=2, default=str)

print("\nModel comparison saved to outputs/model_comparison.csv")

## 11. Select the model using validation data

In [ ]:
# The internal test set is deliberately not used for model selection.
best_model_name = str(model_results.iloc[0]["Model"])
selected_model = fitted_models[best_model_name]

print("Selected model:", best_model_name)
print("Selection criterion: lowest validation MAPE")
print("Validation results:")
print(model_results.iloc[0][[
    "Validation MAPE", "Validation R2", "Validation MAE", "Validation RMSE"
]])

plt.figure(figsize=(10, 5))
plt.barh(model_results["Model"], model_results["Validation MAPE"])
plt.gca().invert_yaxis()
plt.xlabel("Validation MAPE (lower is better)")
plt.title("Candidate Model Comparison")
plt.tight_layout()
plt.show()

## 12. Evaluate once on the untouched internal test partition

This score is the final holdout estimate for the selected modelling procedure. Do not use it to tune parameters or repeatedly select models.

In [ ]:
internal_test_prediction = np.maximum(selected_model.predict(X_internal_test), 0.0)
internal_test_metrics = calculate_metrics(y_internal_test, internal_test_prediction)

internal_test_results = pd.DataFrame([{
    "Selected model": best_model_name,
    "Internal Test MAPE": internal_test_metrics["MAPE"],
    "Internal Test R2": internal_test_metrics["R2"],
    "Internal Test MAE": internal_test_metrics["MAE"],
    "Internal Test RMSE": internal_test_metrics["RMSE"],
}])
display(internal_test_results)
internal_test_results.to_csv(OUTPUT_DIR / "internal_test_metrics.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_internal_test, internal_test_prediction, alpha=0.45)
low = min(float(y_internal_test.min()), float(internal_test_prediction.min()))
high = max(float(y_internal_test.max()), float(internal_test_prediction.max()))
axes[0].plot([low, high], [low, high], linestyle="--")
axes[0].set_xlabel("Actual pressure (bar)")
axes[0].set_ylabel("Predicted pressure (bar)")
axes[0].set_title("Actual vs Predicted – Internal Test")

residuals = np.asarray(y_internal_test) - internal_test_prediction
axes[1].hist(residuals, bins=40)
axes[1].set_xlabel("Residual (actual - predicted)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Internal Test Residual Distribution")
plt.tight_layout()
plt.show()

print("Internal test metrics saved to outputs/internal_test_metrics.csv")

## 13. Refit the selected procedure on all eligible labelled data

In [ ]:
# After completing the holdout evaluation, refit the chosen model on all cleaned labelled data
# before generating predictions for the unlabelled Kaggle competition test dataset.
full_X = X.copy()
full_y = y.copy()

if REMOVE_TARGET_IQR_OUTLIERS:
    full_q1 = full_y.quantile(0.25)
    full_q3 = full_y.quantile(0.75)
    full_iqr = full_q3 - full_q1
    full_lower = full_q1 - IQR_MULTIPLIER * full_iqr
    full_upper = full_q3 + IQR_MULTIPLIER * full_iqr
    full_keep = full_y.between(full_lower, full_upper)
    print(f"Final refit: removing {int((~full_keep).sum()):,} target-IQR outliers from labelled training data.")
    full_X = full_X.loc[full_keep].reset_index(drop=True)
    full_y = full_y.loc[full_keep].reset_index(drop=True)

full_X = cast_categorical(full_X, FORCE_CATEGORICAL)
X_test_final = cast_categorical(X_test.copy(), FORCE_CATEGORICAL)
final_model = clone(selected_model)
final_model.fit(full_X, full_y)

competition_predictions = np.maximum(final_model.predict(X_test_final), 0.0)
if not np.isfinite(competition_predictions).all():
    raise ValueError("Final predictions contain NaN or infinite values.")

print("Competition predictions generated:", len(competition_predictions))
print("Prediction range:", float(competition_predictions.min()), "to", float(competition_predictions.max()))

## 14. Create the Kaggle prediction file

In [ ]:
# Prefer the supplied sample submission format when it is available.
if SAMPLE_SUBMISSION_PATH.exists():
    sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
    if len(sample_submission) != len(test):
        raise ValueError("sample_prediction.csv row count does not match test.csv.")
    if ID_COLUMN in sample_submission.columns and not np.array_equal(
        sample_submission[ID_COLUMN].to_numpy(), test_ids.to_numpy()
    ):
        raise ValueError("Sample submission IDs do not match test.csv order.")

    prediction_columns = [c for c in sample_submission.columns if c != ID_COLUMN]
    if len(prediction_columns) != 1:
        raise ValueError(
            "Expected exactly one prediction column besides ID in sample_prediction.csv."
        )
    submission = sample_submission.copy()
    submission[prediction_columns[0]] = competition_predictions
else:
    submission = pd.DataFrame({
        ID_COLUMN: test_ids.to_numpy(),
        TARGET: competition_predictions,
    })

prediction_path = OUTPUT_DIR / "prediction.csv"
submission.to_csv(prediction_path, index=False)
print("Saved:", prediction_path)
display(submission.head())
print("Submission shape:", submission.shape)

# Download automatically in Google Colab; do nothing on local Jupyter/VS Code.
try:
    from google.colab import files
    files.download(str(prediction_path))
except ImportError:
    print("Not running in Google Colab; prediction file is saved locally.")

## 15. Save reproducibility summary

In [ ]:
summary = {
    "random_state": RANDOM_STATE,
    "target_column": TARGET,
    "cleaned_labelled_rows": int(len(train)),
    "competition_test_rows": int(len(test)),
    "row_missing_threshold": ROW_MISSING_THRESHOLD,
    "remove_target_iqr_outliers": REMOVE_TARGET_IQR_OUTLIERS,
    "iqr_multiplier": IQR_MULTIPLIER,
    "train_rows_after_outlier_treatment": int(len(X_train)),
    "validation_rows": int(len(X_val)),
    "internal_test_rows": int(len(X_internal_test)),
    "cv_folds": CV_FOLDS,
    "search_iterations_per_tuned_model": N_ITER_SEARCH,
    "selected_model": best_model_name,
    "validation_metrics": {
        "MAPE": float(model_results.iloc[0]["Validation MAPE"]),
        "R2": float(model_results.iloc[0]["Validation R2"]),
        "MAE": float(model_results.iloc[0]["Validation MAE"]),
        "RMSE": float(model_results.iloc[0]["Validation RMSE"]),
    },
    "internal_test_metrics": {k: float(v) for k, v in internal_test_metrics.items()},
}

with open(OUTPUT_DIR / "run_summary.json", "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)

print("Run summary saved to outputs/run_summary.json")